# 🔬 Notebook 3: Reminder / Alert — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/reminder-alert
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Hashed hierarchical time wheel

For sub-second accuracy with millions of timers, a priority queue gets slow. A **time wheel** slots timers into buckets by fire time mod N; the dispatcher ticks one bucket per interval.

Below: a simple wheel with 60 buckets (one per second).

In [ ]:
import time

class TimeWheel:
    def __init__(self, size=60):
        self.size = size
        self.slots = [[] for _ in range(size)]
        self.tick = 0

    def schedule(self, delay_s, cb):
        slot = (self.tick + int(delay_s)) % self.size
        self.slots[slot].append(cb)

    def advance(self):
        self.tick = (self.tick + 1) % self.size
        due = self.slots[self.tick]
        self.slots[self.tick] = []
        for cb in due: cb()

w = TimeWheel()
w.schedule(2, lambda: print("2s reminder"))
w.schedule(5, lambda: print("5s reminder"))
for i in range(6):
    w.advance()

## Deep dive 2

### At-least-once + idempotency

Workers can crash mid-delivery, so we may fire twice. We give each reminder a stable **delivery id** and receivers dedupe on that id.

In [ ]:
delivered = set()

def deliver(delivery_id, message):
    if delivery_id in delivered:
        return "duplicate"
    delivered.add(delivery_id)
    return f"sent: {message}"

print(deliver("r1-a", "take pills"))
print(deliver("r1-a", "take pills"))   # same delivery_id → deduped
print(deliver("r2-a", "meeting"))

## Closing thoughts

- Sharding by `user_id` or `hash(fire_at)` spreads the poll load.
- `FOR UPDATE SKIP LOCKED` makes the scheduler trivially parallel.
- Always design receivers to be **idempotent** — at-least-once is the default reality.